# ⛓️ Day 02a: Backpropagation — How Networks Actually Learn

---

## 🎯 What You'll Master Today
- Chain rule intuition
- Backprop step-by-step (by hand)
- Full backprop in NumPy
- Solve XOR from scratch

**Goal:** Implement full forward + backward pass from memory.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  BACKPROP — The Core Algorithm of Deep Learning                  ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Forward:  X → Z1 → A1 → Z2 → A2 → Loss                      ║
║  Backward: Loss → dA2 → dZ2 → dW2 → dA1 → dZ1 → dW1         ║
║                                                                   ║
║  Chain Rule:                                                     ║
║    ∂Loss/∂W1 = ∂Loss/∂A2 · ∂A2/∂Z2 · ∂Z2/∂A1 · ∂A1/∂Z1 ·   ║
║                ∂Z1/∂W1                                           ║
║                                                                   ║
║  Key Gradients:                                                  ║
║    dZ_output = A - y           (for sigmoid + BCE)              ║
║    dW = A_prev.T @ dZ          (weight gradient)                ║
║    db = sum(dZ)                 (bias gradient)                 ║
║    dA_prev = dZ @ W.T           (pass gradient back)           ║
║    dZ = dA * relu_derivative     (through activation)           ║
║                                                                   ║
║  Update: W = W - lr * dW                                        ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# 1. Chain Rule — Intuition
# ============================================================

# f(x) = (3x + 2)²
# Let u = 3x + 2, f = u²
# df/dx = df/du · du/dx = 2u · 3 = 6(3x+2)

x = 1.0
# Manual
u = 3*x + 2
df_du = 2*u
du_dx = 3
df_dx = df_du * du_dx
print(f"Chain rule: df/dx = {df_dx}")

# Numerical verification
eps = 1e-7
f = lambda x: (3*x + 2)**2
numerical = (f(x + eps) - f(x - eps)) / (2 * eps)
print(f"Numerical:  df/dx = {numerical:.6f}")

print("\n💡 Backprop IS the chain rule, applied layer by layer.")

In [ ]:
# ============================================================
# 2. Activation Derivatives
# ============================================================

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)    # elegant: σ'(z) = σ(z)(1-σ(z))

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)  # 1 if z>0, else 0

z = np.array([-2, -1, 0, 1, 2])
print(f"z:             {z}")
print(f"sigmoid'(z):   {sigmoid_derivative(z).round(4)}")
print(f"relu'(z):      {relu_derivative(z)}")

print("\n💡 sigmoid': max at z=0 (0.25), vanishes at extremes → vanishing gradient!")
print("   relu': either 0 or 1 → no vanishing gradient (but dead neurons at z<0)")

In [ ]:
# ============================================================
# 3. Full Neural Network with Backprop — XOR Solver
# ============================================================

class NeuralNetXOR:
    """2 → 4 → 1 network to solve XOR."""
    
    def __init__(self):
        np.random.seed(42)
        # He initialization for ReLU hidden layer
        self.W1 = np.random.randn(2, 4) * np.sqrt(2/2)  # (2, 4)
        self.b1 = np.zeros(4)
        # Xavier for sigmoid output
        self.W2 = np.random.randn(4, 1) * np.sqrt(1/4)  # (4, 1)
        self.b2 = np.zeros(1)
    
    def forward(self, X):
        # Layer 1: ReLU
        self.Z1 = X @ self.W1 + self.b1        # (batch, 4)
        self.A1 = relu(self.Z1)                  # (batch, 4)
        # Output: Sigmoid
        self.Z2 = self.A1 @ self.W2 + self.b2   # (batch, 1)
        self.A2 = sigmoid(self.Z2)               # (batch, 1)
        return self.A2
    
    def compute_loss(self, y_pred, y_true):
        """Binary cross-entropy loss."""
        eps = 1e-15
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1-y_true) * np.log(1-y_pred))
    
    def backward(self, X, y):
        m = X.shape[0]  # batch size
        
        # Output layer gradient
        # For sigmoid + BCE: dZ2 = A2 - y (clean!)
        dZ2 = self.A2 - y                        # (batch, 1)
        dW2 = (self.A1.T @ dZ2) / m              # (4, 1)
        db2 = np.mean(dZ2, axis=0)                # (1,)
        
        # Hidden layer gradient
        dA1 = dZ2 @ self.W2.T                     # (batch, 4)
        dZ1 = dA1 * relu_derivative(self.Z1)      # (batch, 4)
        dW1 = (X.T @ dZ1) / m                     # (2, 4)
        db1 = np.mean(dZ1, axis=0)                # (4,)
        
        return dW1, db1, dW2, db2
    
    def update(self, dW1, db1, dW2, db2, lr=0.1):
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2

# XOR data
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([[0], [1], [1], [0]])

# Train
nn = NeuralNetXOR()
losses = []

for epoch in range(5000):
    # Forward
    y_pred = nn.forward(X)
    loss = nn.compute_loss(y_pred, y)
    losses.append(loss)
    
    # Backward
    dW1, db1, dW2, db2 = nn.backward(X, y)
    nn.update(dW1, db1, dW2, db2, lr=1.0)
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss:.6f}")

# Results
print(f"\nFinal predictions:")
for i in range(4):
    print(f"  {X[i]} → {y_pred[i][0]:.4f} (expected {y[i][0]})")

In [ ]:
# ============================================================
# 4. Gradient Checking — Verify Your Backprop is Correct
# ============================================================

def gradient_check(nn, X, y, epsilon=1e-7):
    """Compare analytical gradients with numerical gradients."""
    # Get analytical gradients
    nn.forward(X)
    dW1, db1, dW2, db2 = nn.backward(X, y)
    
    # Check W1 gradients (sample a few)
    print("Gradient Check (W1):")
    for i in range(min(3, nn.W1.shape[0])):
        for j in range(min(3, nn.W1.shape[1])):
            # Numerical gradient
            old_val = nn.W1[i, j]
            
            nn.W1[i, j] = old_val + epsilon
            loss_plus = nn.compute_loss(nn.forward(X), y)
            
            nn.W1[i, j] = old_val - epsilon
            loss_minus = nn.compute_loss(nn.forward(X), y)
            
            nn.W1[i, j] = old_val  # restore
            
            numerical = (loss_plus - loss_minus) / (2 * epsilon)
            analytical = dW1[i, j]
            
            diff = abs(numerical - analytical) / (abs(numerical) + abs(analytical) + 1e-8)
            status = '✅' if diff < 1e-5 else '❌'
            print(f"  W1[{i},{j}]: analytical={analytical:.8f}, numerical={numerical:.8f}, diff={diff:.2e} {status}")

nn_check = NeuralNetXOR()
gradient_check(nn_check, X, y)

print("\n💡 Gradient checking is ESSENTIAL when implementing backprop from scratch.")
print("   diff < 1e-5 → correct. diff > 1e-3 → bug!")

In [ ]:
# ============================================================
# 5. Backprop Formulas — Quick Reference
# ============================================================

print("""
BACKPROP FORMULAS — Write From Memory
══════════════════════════════════════

FORWARD:
  Z[l] = A[l-1] @ W[l] + b[l]
  A[l] = activation(Z[l])

BACKWARD (output layer, sigmoid + BCE):
  dZ[L] = A[L] - y

BACKWARD (hidden layers):
  dA[l]   = dZ[l+1] @ W[l+1].T
  dZ[l]   = dA[l] * activation_derivative(Z[l])

GRADIENTS:
  dW[l] = (A[l-1].T @ dZ[l]) / m
  db[l] = mean(dZ[l], axis=0)

UPDATE:
  W[l] = W[l] - lr * dW[l]
  b[l] = b[l] - lr * db[l]

SHAPES:
  dZ[l]: (batch, neurons_l)   — same as Z[l]
  dW[l]: (neurons_{l-1}, neurons_l) — same as W[l]
  db[l]: (neurons_l,)        — same as b[l]
""")

---

## 🗺️ Backpropagation Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  BACKPROP — The Algorithm                                        ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  1. Forward pass → compute all Z[l], A[l]                      ║
║  2. Compute loss                                                 ║
║  3. Backward pass → compute dZ, dW, db for each layer          ║
║  4. Update weights: W -= lr * dW                                ║
║  5. Repeat                                                       ║
║                                                                   ║
║  KEY INSIGHT:                                                    ║
║    dZ_output = A - y  (sigmoid + BCE simplification)            ║
║    dW = A_prev.T @ dZ / m                                       ║
║    dA_prev = dZ @ W.T                                           ║
║                                                                   ║
║  DEBUGGING:                                                      ║
║    Gradient checking: (f(x+ε) - f(x-ε)) / 2ε                  ║
║    Compare with analytical gradient. diff < 1e-5 = correct     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Explain backpropagation | Use chain rule to compute ∂Loss/∂W for each layer, from output back to input. Then update W -= lr * gradient |
| 2 | Why do we need backprop? | Computing ∂L/∂W directly is O(n²) per param. Backprop reuses intermediate results → O(n) |
| 3 | What is vanishing gradient? | Sigmoid derivative max = 0.25. Multiply across many layers → gradient → 0. Solution: ReLU, skip connections |
| 4 | What is gradient checking? | Compare analytical vs numerical gradient: (f(x+ε)-f(x-ε))/2ε. Verifies backprop implementation |
| 5 | dZ = A - y comes from where? | Derivative of sigmoid + BCE loss simplifies to this clean form. Only for output layer |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Backprop = chain rule applied layer by layer         │
## │  • Output gradient: dZ = A - y (sigmoid + BCE)         │
## │  • Weight gradient: dW = A_prev.T @ dZ / m             │
## │  • Pass back: dA_prev = dZ @ W.T                       │
## │  • Always verify with gradient checking                 │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Activations & Loss Functions** — ReLU, softmax, cross-entropy deep dive